<a href="https://colab.research.google.com/github/fariahasan00/Bangla-SLM-498R/blob/main/BanglaSLM15M.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import shutil
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

DRIVE_TRAIN_BIN = "/content/drive/MyDrive/Bangla_SLM_15M/data/train.bin"
DRIVE_VAL_BIN = "/content/drive/MyDrive/Bangla_SLM_15M/data/val.bin"

LOCAL_DATA_DIR = "/content/local_data"
LOCAL_TRAIN_BIN = os.path.join(LOCAL_DATA_DIR, "train.bin")
LOCAL_VAL_BIN = os.path.join(LOCAL_DATA_DIR, "val.bin")

BLOCK_SIZE = 512
BATCH_SIZE = 32
NUM_WORKERS = 2  # Colab CPU workers

def prepare_local_data(drive_path: str, local_path: str) -> str:
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    if not os.path.exists(local_path):
        print(f"Copying {drive_path} -> {local_path} to local NVMe SSD...")
        shutil.copyfile(drive_path, local_path)
        print("Copy complete!")
    return local_path

train_file = prepare_local_data(DRIVE_TRAIN_BIN, LOCAL_TRAIN_BIN)
val_file = prepare_local_data(DRIVE_VAL_BIN, LOCAL_VAL_BIN)


class OptimizedMemMapDataset(Dataset):
    def __init__(self, bin_path: str, block_size: int = 512, dtype=np.uint16):
        super().__init__()
        self.bin_path = bin_path
        self.block_size = block_size
        self.dtype = dtype

        file_bytes = os.path.getsize(self.bin_path)
        self.total_tokens = file_bytes // np.dtype(self.dtype).itemsize
        self.num_samples = (self.total_tokens - 1) // self.block_size
        self.data = None  # Lazy handle per worker process

    def __len__(self) -> int:
        return self.num_samples

    def __getitem__(self, idx: int):
        if self.data is None:
            self.data = np.memmap(self.bin_path, dtype=self.dtype, mode='r')

        start_idx = idx * self.block_size
        end_idx = start_idx + self.block_size + 1

        # Direct memory slice zero-copy to PyTorch Tensor
        chunk = torch.from_numpy(np.array(self.data[start_idx:end_idx])).long()

        # Input (x) and Target (y) shift
        return chunk[:-1], chunk[1:]


train_dataset = OptimizedMemMapDataset(train_file, block_size=BLOCK_SIZE)
val_dataset = OptimizedMemMapDataset(val_file, block_size=BLOCK_SIZE)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,             # Randomizes sequence block order each epoch
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,            # Deterministic evaluation order
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    drop_last=False
)

if __name__ == "__main__":
    for x_batch, y_batch in train_loader:
        print("\n--- Sanity Check ---")
        print(f"Input batch shape (X) : {x_batch.shape}")
        print(f"Target batch shape (Y): {y_batch.shape}")
        assert torch.equal(x_batch[0, 1:], y_batch[0, :-1]), "Target shift mismatch!"
        print("Shift verification    : Passed (y[t] == x[t+1])")
        break

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



--- Sanity Check ---
Input batch shape (X) : torch.Size([32, 512])
Target batch shape (Y): torch.Size([32, 512])
Shift verification    : Passed (y[t] == x[t+1])


In [4]:
import sentencepiece as spm

# 1. Load your SentencePiece tokenizer model
sp_model_path = (
    "/content/drive/MyDrive/Bangla_SLM_15M/data/tokenizer/bangla_sp.model"
)
sp = spm.SentencePieceProcessor()
sp.load(sp_model_path)

# 2. Fetch one batch from train_loader and decode
for x_batch, y_batch in train_loader:
    # Get token IDs for the first sample in the batch
    x_ids = x_batch[0].tolist()
    y_ids = y_batch[0].tolist()

    # Decode integer IDs back into raw Bangla text
    x_text = sp.decode(x_ids)
    y_text = sp.decode(y_ids)

    print("=================== BANGLA INPUT (X) ===================")
    print(x_text[:300])  # Printing first 300 characters

    print("\n=================== BANGLA TARGET (Y) ===================")
    print(y_text[:300])

    break

=================== BANGLA INPUT (X) ===================
পেয়ালায় চিনির দানা গলে গেল, এমনিভাবে সেই চাউনি গলে গেল আমার ভেতর। তার ক্ষীণ, মৃদু হাসির সঙ্গে আমি তার গলা শুনতে পেলুম। ‘চাচা! তুমি কি এক্ষুনি কুয়েত থেকে এলে?’ তার কণ্ঠস্বর কী-রকম যেন ভেঙে গেল তার গলায়, দুই হাতে ভর দিয়ে কোনোমতে সে উঠে বসল, গলাটা বাড়িয়ে দিল আমার দিকে, আমি হালকাভাবে তার পিঠ চাপড

=================== BANGLA TARGET (Y) ===================
য়ালায় চিনির দানা গলে গেল, এমনিভাবে সেই চাউনি গলে গেল আমার ভেতর। তার ক্ষীণ, মৃদু হাসির সঙ্গে আমি তার গলা শুনতে পেলুম। ‘চাচা! তুমি কি এক্ষুনি কুয়েত থেকে এলে?’ তার কণ্ঠস্বর কী-রকম যেন ভেঙে গেল তার গলায়, দুই হাতে ভর দিয়ে কোনোমতে সে উঠে বসল, গলাটা বাড়িয়ে দিল আমার দিকে, আমি হালকাভাবে তার পিঠ চাপড়ে
